# Evaluating GenAI Responses
### Practice Notebook

This notebook has no live LLM API calls wired in -- every "model output" is
a pre-written example, so the focus stays entirely on *evaluation*: how do
you decide whether a given response is actually good?


## 1. Why evaluation is hard for generative outputs

Unlike a spam classifier (right/wrong, one correct label), a generative
response can be "correct" in many different valid phrasings, and "wrong" in
subtle ways -- plausible-sounding but factually off, rather than obviously
broken. There's often no single ground-truth answer to compare against, and
quality is multi-dimensional: a response can be accurate but unhelpful, or
friendly but wrong.

Let's see why naive string-matching metrics fail here.


In [1]:
def naive_exact_match(response: str, reference: str) -> bool:
    """The crudest possible 'evaluation': does the response exactly match
    a reference answer?"""
    return response.strip().lower() == reference.strip().lower()

reference = "The capital of Bharat is Delhi."
candidate_responses = [
    "The capital of Bharat is Delhi.",              # identical
    "Delhi is the capital of Bharat.",               # same meaning, different wording
    "Bharat's capital city is Delhi, a major business hub in Asia.",  # correct + extra detail
    "The capital of Bharat is Nagpur.",                # confidently WRONG
]

for resp in candidate_responses:
    print(f"exact_match={naive_exact_match(resp, reference)!s:5}  '{resp}'")


exact_match=True   'The capital of Bharat is Delhi.'
exact_match=False  'Delhi is the capital of Bharat.'
exact_match=False  'Bharat's capital city is Delhi, a major business hub in Asia.'
exact_match=False  'The capital of Bharat is Nagpur.'


**Exercise 1.1:** Notice `naive_exact_match` marks the second and third
responses as "wrong" even though they're correct (and the third is arguably
*more* helpful than the reference). Meanwhile it has no special mechanism to
flag the fourth response as dangerous -- it just says False, the same as the
other paraphrases. What does this tell you about why word-overlap metrics
like BLEU/ROUGE (built for tasks with one fixed reference answer) are a poor
fit for evaluating open-ended generative responses?


## 2. A structured rubric: relevance, coherence, correctness, helpfulness

Instead of a single pass/fail judgment, score each response along four
independent dimensions. Let's build a simple structured rubric and manually
score a few example responses, to feel out how these dimensions can diverge
from each other.

**Relevance:** Measures how directly the LLM response addresses the user's prompt without introducing off-topic information.

**Coherence:** Evaluates the structural flow, logical organization, and overall readability of the generated text.

**Correctness:** Verifies that the output is factually accurate, free of hallucinations, and aligns with ground-truth data.

**Helpfulness:** Assesses how effectively the response solves the user's underlying problem in a clear and actionable manner.


In [4]:
RUBRIC_DIMENSIONS = ["relevance", "coherence", "correctness", "helpfulness"]

def make_score(relevance: int, coherence: int, correctness: int, helpfulness: int) -> dict:
    """Each dimension scored 1 (poor) to 5 (excellent)."""
    return {"relevance": relevance, "coherence": coherence,
            "correctness": correctness, "helpfulness": helpfulness}

examples = [
    {
        "question": "How do I reset my router?",
        "response": "Unplug the router, wait 10 seconds, then plug it back in. "
                     "If that doesn't work, hold the reset button for 10 seconds.",
        # Manually assigned scores for this worked example:
        "human_score": make_score(relevance=5, coherence=5, correctness=5, helpfulness=5),
    },
    {
        "question": "How do I reset my router?",
        "response": "Routers are networking devices that connect your home devices "
                     "to the internet. They were invented to replace older hub-based "
                     "systems and have evolved significantly since the 1990s.",
        # Relevant topic, coherent, factually fine, but doesn't answer the question:
        "human_score": make_score(relevance=2, coherence=5, correctness=5, helpfulness=1),
    },
    {
        "question": "How do I reset my router?",
        "response": "just unplug it. reset button. wait. plug in. shud work. if not "
                     "try again idk maybe call support or smth",
        # Actually contains the right idea, but poorly structured and hedged:
        "human_score": make_score(relevance=4, coherence=2, correctness=4, helpfulness=3),
    },
]

for ex in examples:
    print(f"Q: {ex['question']}")
    print(f"A: {ex['response']}")
    print(f"Scores: {ex['human_score']}\n")


Q: How do I reset my router?
A: Unplug the router, wait 10 seconds, then plug it back in. If that doesn't work, hold the reset button for 10 seconds.
Scores: {'relevance': 5, 'coherence': 5, 'correctness': 5, 'helpfulness': 5}

Q: How do I reset my router?
A: Routers are networking devices that connect your home devices to the internet. They were invented to replace older hub-based systems and have evolved significantly since the 1990s.
Scores: {'relevance': 2, 'coherence': 5, 'correctness': 5, 'helpfulness': 1}

Q: How do I reset my router?
A: just unplug it. reset button. wait. plug in. shud work. if not try again idk maybe call support or smth
Scores: {'relevance': 4, 'coherence': 2, 'correctness': 4, 'helpfulness': 3}



**Exercise 1.2:** The second example scores low on helpfulness (1) despite
scoring high on coherence and correctness (5 each) -- it's a well-written,
accurate paragraph that simply never answers the question. Write one more
example response to "How do I reset my router?" that would score **high on
correctness but low on relevance** (a different combination than any example
above), and assign it scores using `make_score`. This is the core reason a
single overall "quality" number is often insufficient: two responses with
the same average score can be bad in completely different ways.


## 3. Human evaluation vs. automated evaluation

Human evaluation is the gold standard for nuance, but slow and expensive.
Automated evaluation is fast and repeatable but can miss nuance or be
gamed. Let's build a **very simple automated proxy** for two of the four
rubric dimensions, then compare it against the human scores above to see
where it agrees and disagrees.


In [5]:
def automated_relevance_proxy(question: str, response: str) -> int:
    """Extremely crude automated relevance check: does the response share
    enough keywords with the question? Real automated evaluation would use
    embedding similarity as we saw in Week2 or an LLM-as-judge (later to introduce)
    instead of keyword overlap -- this stub exists purely to demonstrate
    the *category* of automated metric, cheaply and offline.
    """
    q_words = set(w.lower() for w in question.split() if len(w) > 3)
    r_words = set(w.lower() for w in response.split() if len(w) > 3)
    overlap = len(q_words & r_words)
    if overlap == 0:
        return 1
    elif overlap == 1:
        return 3
    else:
        return 5

def automated_coherence_proxy(response: str) -> int:
    """Extremely crude automated coherence check: average sentence length
    and capitalization as a stand-in for 'well-structured'. Real automated
    coherence checks are far more sophisticated (or use an LLM-as-judge).
    """
    sentences = [s.strip() for s in response.split(".") if s.strip()]
    if not sentences:
        return 1
    proper_capitalization = sum(1 for s in sentences if s[0].isupper()) / len(sentences)
    return 5 if proper_capitalization == 1.0 else (3 if proper_capitalization >= 0.5 else 1)

for ex in examples:
    auto_relevance = automated_relevance_proxy(ex["question"], ex["response"])
    auto_coherence = automated_coherence_proxy(ex["response"])
    print(f"Response: '{ex['response'][:50]}...'")
    print(f"  Human relevance={ex['human_score']['relevance']}  vs  Automated relevance={auto_relevance}")
    print(f"  Human coherence={ex['human_score']['coherence']}  vs  Automated coherence={auto_coherence}\n")


Response: 'Unplug the router, wait 10 seconds, then plug it b...'
  Human relevance=5  vs  Automated relevance=3
  Human coherence=5  vs  Automated coherence=5

Response: 'Routers are networking devices that connect your h...'
  Human relevance=2  vs  Automated relevance=1
  Human coherence=5  vs  Automated coherence=5

Response: 'just unplug it. reset button. wait. plug in. shud ...'
  Human relevance=4  vs  Automated relevance=3
  Human coherence=2  vs  Automated coherence=1



**Exercise 1.3:** Where does the automated proxy agree with the human
score, and where does it disagree? For any disagreement, explain in one
sentence *why* the crude automated heuristic got it wrong -- this is exactly
the kind of gap that motivates using a real LLM-as-judge (Part 4) instead of
simple heuristics for anything beyond a first-pass sanity check.


## 4. LLM-as-judge (conceptual + optional real call)

A more capable automated approach: give a separate LLM call the question,
the response, and the rubric, and ask it to score the response directly.
Let's build the *shape* of this with a stub, then show the real version.


In [6]:
JUDGE_PROMPT_TEMPLATE = '''You are an expert evaluator. Score the following
response to a user's question on four dimensions, each from 1 (poor) to 5
(excellent): relevance, coherence, correctness, helpfulness.

Question: {question}
Response: {response}

Return your answer as a JSON object with keys: relevance, coherence,
correctness, helpfulness, and a one-sentence justification for each.
'''

def llm_judge_stub(question: str, response: str) -> dict:
    """# TODO: replace with a real LLM call using JUDGE_PROMPT_TEMPLATE.
    This stub just echoes a plausible-looking but fake judgment so the
    notebook's structure is runnable offline.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, response=response)
    return {"relevance": 3, "coherence": 3, "correctness": 3, "helpfulness": 3,
            "note": "(stub) replace with a real LLM call for a real judgment",
            "prompt_used": prompt}

result = llm_judge_stub(examples[0]["question"], examples[0]["response"])
print(result["prompt_used"])
print()
print("Stub judgment:", {k: v for k, v in result.items() if k in RUBRIC_DIMENSIONS})


You are an expert evaluator. Score the following
response to a user's question on four dimensions, each from 1 (poor) to 5
(excellent): relevance, coherence, correctness, helpfulness.

Question: How do I reset my router?
Response: Unplug the router, wait 10 seconds, then plug it back in. If that doesn't work, hold the reset button for 10 seconds.

Return your answer as a JSON object with keys: relevance, coherence,
correctness, helpfulness, and a one-sentence justification for each.


Stub judgment: {'relevance': 3, 'coherence': 3, 'correctness': 3, 'helpfulness': 3}


**Exercise 1.4 (mini deliverable):** Pick 3 question/response pairs of your
own (write both the question and a response yourself -- try to include at
least one deliberately flawed response, similar to the router examples
above). For each: (a) assign your own human rubric scores using
`make_score`, (b) run them through `automated_relevance_proxy` and
`automated_coherence_proxy`, and (c) write out what a real LLM-as-judge
prompt for them would look like using `JUDGE_PROMPT_TEMPLATE`. Compare all
three approaches and write 2-3 sentences on which you'd trust most for a
real production system, and why.
